# 🎨 Agentic Design Patterns with Azure AI Foundry Models (Python)

## 📋 Learning Objectives

This notebook demonstrates essential design patterns for building intelligent agents using the Microsoft Agent Framework with Azure AI Foundry (Azure OpenAI) model integration. You'll learn proven patterns and architectural approaches that make agents more robust, maintainable, and effective.

**Core Design Patterns Covered:**
- 🏗️ **Agent Factory Pattern**: Standardized agent creation and configuration
- 🔧 **Tool Registry Pattern**: Organized approach to managing agent capabilities
- 🧵 **Conversation Management**: Effective patterns for multi-turn interactions
- 🔄 **Response Processing**: Best practices for handling agent outputs

## 🎯 Key Architectural Concepts

### Design Principles
- **Separation of Concerns**: Clear boundaries between agent logic, tools, and configuration
- **Composability**: Building complex agents from reusable components
- **Extensibility**: Patterns that allow easy addition of new capabilities
- **Testability**: Design for easy unit testing and validation

### Azure AI Foundry Integration
- **Model Deployments**: Working with model deployment names (e.g., `gpt-4o`, `gpt-4o-mini`, `gpt-4.1-mini`) rather than raw model IDs
- **Endpoint Usage**: Calling Azure AI Foundry (Azure OpenAI) endpoints using Azure identity or API keys
- **Model Selection**: Choosing appropriate deployments for different use cases
- **Rate Limiting & Quotas**: Handling service constraints gracefully
- **Error Recovery**: Robust error handling and retry patterns

## 🔧 Technical Architecture

### Core Components
- **Microsoft Agent Framework**: Python implementation with Azure AI Foundry model support
- **Azure AI Foundry / Azure OpenAI API**: Access to state-of-the-art language models via secure endpoints
- **OpenAI-Compatible Client Pattern**: Standardized API interaction patterns (adapter approach)
- **Environment Configuration**: Secure and flexible configuration management

### Design Pattern Benefits
- **Maintainability**: Clear code organization and structure
- **Scalability**: Patterns that grow with your application needs
- **Reliability**: Proven approaches that handle edge cases
- **Performance**: Efficient resource utilization and API usage

## ⚙️ Prerequisites & Setup

**Required Dependencies:**
```bash
pip install agent-framework-core -U
```

**Environment Configuration (.env file):**
```env
AZURE_AI_FOUNDRY_API_KEY=your_azure_openai_key
AZURE_AI_FOUNDRY_ENDPOINT=https://your-resource-name.openai.azure.com
AZURE_AI_FOUNDRY_MODEL=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-08-01-preview
```
(Or configure Azure CLI / Managed Identity and use credential-based access.)

**Azure AI Foundry Access:**
- Azure subscription with Azure OpenAI access approved (if required)
- Deployed model (deployment name referenced by `AZURE_AI_FOUNDRY_MODEL`)
- Properly scoped API key OR Azure Active Directory auth
- Awareness of quota and rate limits for your pricing tier

## 📚 Design Pattern Categories

### 1. **Creational Patterns**
- Agent factory and builder patterns
- Configuration management patterns
- Dependency injection for agent services

### 2. **Behavioral Patterns**
- Tool execution and orchestration
- Conversation flow management  
- Response processing and formatting

### 3. **Integration Patterns**
- Azure AI Foundry endpoint integration
- Error handling and retry logic
- Resource management and cleanup

## 🚀 Best Practices Demonstrated

- **Clean Architecture**: Layered design with clear responsibilities
- **Error Handling**: Comprehensive exception management
- **Configuration**: Environment-based setup for different environments
- **Testing**: Patterns that enable effective unit and integration testing
- **Documentation**: Self-documenting code with clear intent

Ready to explore professional agent design patterns? Let's build something robust! 🌟

In [1]:
# 📦 Import Core Libraries for Agent Design Patterns
import os                     # Environment variable access for configuration management
from random import randint    # Random selection utilities for tool functionality

from dotenv import load_dotenv  # Secure environment configuration loading

In [2]:
# Configure the Azure AI Agent Client
# This connects your agent to the AI model
load_dotenv()
# Get credentials from environment variables
api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
foundry_openai_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL", "gpt-4o")

if not api_key:
    raise ValueError(
        "AZURE_AI_FOUNDRY_API_KEY environment variable is required. "
        "Please set it in your .env file or environment."
    )

from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential
# Azure version using OpenAI-compatible client

chat_client= AzureOpenAIChatClient(
    endpoint=foundry_openai_endpoint,  # Azure OpenAI endpoint
    api_key=api_key,
    deployment_name=model_name  # Your deployment name
)


In [3]:
# 🛠️ Tool Function Design Pattern
# Implements the Strategy Pattern for pluggable agent capabilities.
# Demonstrates separation of business logic from agent orchestration.
def get_random_destination() -> str:
    """Get a random vacation destination.
    
    Patterns illustrated:
    - Strategy Pattern: Interchangeable selection algorithm
    - Repository Pattern: Encapsulated data source
    - Factory Method: Creates destination objects on demand
    
    Returns:
        str: A randomly selected destination.
    """
    destinations = [
        "Barcelona, Spain",
        "Paris, France",
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    return destinations[randint(0, len(destinations) - 1)]

In [5]:
AGENT_NAME ="TravelAgent"

AGENT_INSTRUCTIONS = """You are a helpful AI Agent that can help plan vacations for customers.

Important: When users specify a destination, always plan for that location. Only suggest random destinations when the user hasn't specified a preference.

When the conversation begins, introduce yourself with this message:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

Always prioritize user preferences. If they mention a specific destination like "Bali" or "Paris," focus your planning on that location rather than suggesting alternatives.
"""

In [ ]:
# Create the agent with Azure AI Foundry-backed chat client and a tool.
agent = ChatAgent(
        name = AGENT_NAME,
        chat_client=chat_client,
        instructions=AGENT_INSTRUCTIONS,
        tools=[get_random_destination]
)

In [7]:
thread = agent.get_new_thread()

In [9]:
response1 = await agent.run("Plan me a day trip",thread= thread)

In [10]:

last_message = response1.messages[-1]
text_content = last_message.contents[0].text
print("Travel plan:")
print(text_content)

Travel plan:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

To get started, could you tell me which destination or type of location you'd like for your day trip? Would you prefer a beach, mountain, city tour, historical places, or something adventurous?


In [11]:
response2 = await agent.run("I don't like that destination. Plan me another vacation.",thread= thread)

[2025-11-04 00:05:04 - c:\Adobe\ai-agents\.venv\Lib\site-packages\agent_framework\_clients.py:609 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


In [12]:
last_message = response2.messages[-1]
text_content = last_message.contents[0].text
print("Change plan:")
print(text_content)

Change plan:
It seems you’d like me to come up with suggestions for a vacation, but you haven’t yet shared a destination preference. Would you like me to choose a random vacation destination for you, or do you have some interests in mind (beaches, mountains, historical sites, etc.)?

Let me know your preferences so I can create the perfect trip!


In [13]:
# 🔧 Implementing Space Principle: Enhanced Travel Agent with Connections

# First, let's create a knowledge graph for travel connections
class TravelConnectionGraph:
    def __init__(self):
        self.connections = {
            # Events happening in destinations
            "Paris, France": {
                "events": ["Paris Fashion Week (Sept)", "Louvre Night Tours", "Seine River Festival"],
                "local_experts": ["Marie (food blogger)", "Jacques (history guide)"],
                "hidden_gems": ["Covered Passages", "Promenade Plantée", "Père Lachaise Cemetery"]
            },
            "Tokyo, Japan": {
                "events": ["Cherry Blossom Festival", "Tokyo Game Show", "Sumida River Fireworks"],
                "local_experts": ["Yuki (temple guide)", "Hiroshi (food critic)"],
                "hidden_gems": ["Golden Gai", "Yanaka District", "Robot Restaurant"]
            },
            "Barcelona, Spain": {
                "events": ["La Mercè Festival", "Barcelona Beach Festival", "Gaudí Architecture Tours"],
                "local_experts": ["Carlos (architecture expert)", "Isabella (flamenco dancer)"],
                "hidden_gems": ["Park Güell at sunrise", "El Born Cultural Center", "Bunkers del Carmel"]
            }
        }
    
    def get_connections(self, destination: str) -> dict:
        return self.connections.get(destination, {})
    
    def should_suggest_connections(self, destination: str, user_interests: list = None) -> bool:
        """Only suggest connections if they're truly relevant"""
        connections = self.get_connections(destination)
        if not connections:
            return False
        
        # Smart logic: suggest if we have rich connections for this destination
        total_items = len(connections.get("events", [])) + len(connections.get("hidden_gems", []))
        return total_items >= 3

# Create our enhanced travel graph
travel_graph = TravelConnectionGraph()

# Enhanced destination tool that leverages connections
def get_enhanced_destination() -> dict:
    """Get a destination with rich connection data"""
    base_destination = get_random_destination()
    connections = travel_graph.get_connections(base_destination)
    
    return {
        "destination": base_destination,
        "has_connections": bool(connections),
        "connection_count": len(connections.get("events", [])) + len(connections.get("hidden_gems", []))
    }

# Test the enhancement
enhanced_dest = get_enhanced_destination()
print(f"Enhanced destination selection: {enhanced_dest}")

# Show connections for a specific destination
paris_connections = travel_graph.get_connections("Paris, France")
print(f"\nParis connections: {paris_connections}")

Enhanced destination selection: {'destination': 'Cape Town, South Africa', 'has_connections': False, 'connection_count': 0}

Paris connections: {'events': ['Paris Fashion Week (Sept)', 'Louvre Night Tours', 'Seine River Festival'], 'local_experts': ['Marie (food blogger)', 'Jacques (history guide)'], 'hidden_gems': ['Covered Passages', 'Promenade Plantée', 'Père Lachaise Cemetery']}


In [14]:
# 🔧 Implementing Time Principle: Memory-Enhanced Travel Agent

from datetime import datetime

# Memory system for our Travel Agent
class TravelMemory:
    def __init__(self):
        self.user_preferences = {}
        self.conversation_history = []
        self.successful_suggestions = []
        
    def record_interaction(self, user_input: str, agent_response: str, user_feedback: str = None):
        """Record each interaction for learning"""
        interaction = {
            "timestamp": datetime.now().isoformat(),
            "user_input": user_input,
            "agent_response": agent_response,
            "user_feedback": user_feedback
        }
        self.conversation_history.append(interaction)
        
        # Extract preferences from user input
        self._extract_preferences(user_input, user_feedback)
    
    def _extract_preferences(self, user_input: str, feedback: str = None):
        """Learn user preferences from their inputs and feedback"""
        input_lower = user_input.lower()
        
        # Extract destination type preferences
        if "beach" in input_lower:
            self.user_preferences["prefers_beaches"] = True
        if "mountain" in input_lower:
            self.user_preferences["prefers_mountains"] = True
        if "city" in input_lower or "urban" in input_lower:
            self.user_preferences["prefers_cities"] = True
        if "historical" in input_lower or "history" in input_lower:
            self.user_preferences["likes_history"] = True
        
        # Learn from feedback
        if feedback:
            feedback_lower = feedback.lower()
            if "too long" in feedback_lower or "shorter" in feedback_lower:
                self.user_preferences["prefers_concise"] = True
            if "more detail" in feedback_lower:
                self.user_preferences["wants_detail"] = True
    
    def get_relevant_context(self, current_request: str) -> str:
        """Get relevant past context for current request"""
        if not self.conversation_history:
            return ""
        
        # Simple relevance: look for similar keywords in past conversations
        request_words = set(current_request.lower().split())
        relevant_interactions = []
        
        for interaction in self.conversation_history[-5:]:  # Last 5 interactions
            past_words = set(interaction["user_input"].lower().split())
            if request_words.intersection(past_words):
                relevant_interactions.append(interaction)
        
        if relevant_interactions:
            return f"Based on our previous conversations, I remember you were interested in {', '.join(self.user_preferences.keys())}"
        return ""

# Create memory system for our agent
travel_memory = TravelMemory()

# Enhanced instruction adaptation based on learned preferences
def adapt_instructions_with_memory(base_instructions: str, memory: TravelMemory) -> str:
    """Adapt agent instructions based on learned user preferences"""
    adaptations = []
    
    if memory.user_preferences.get("prefers_concise"):
        adaptations.append("Keep responses concise and to the point.")
    if memory.user_preferences.get("wants_detail"):
        adaptations.append("Provide detailed information and context.")
    if memory.user_preferences.get("prefers_beaches"):
        adaptations.append("When suggesting destinations, prioritize coastal and beach locations.")
    if memory.user_preferences.get("likes_history"):
        adaptations.append("Include historical context and cultural sites in recommendations.")
    
    if adaptations:
        return base_instructions + "\n\n# Learned User Preferences:\n" + "\n".join(adaptations)
    return base_instructions

# Test memory system
travel_memory.record_interaction(
    "I want a beach vacation", 
    "I suggest Bali for beautiful beaches",
    "That sounds perfect!"
)

travel_memory.record_interaction(
    "Can you suggest something shorter?",
    "Brief suggestion: Try Nice, France",
    "Much better"
)

print("Learned preferences:", travel_memory.user_preferences)

# Show how instructions adapt
adapted_instructions = adapt_instructions_with_memory(AGENT_INSTRUCTIONS, travel_memory)
print("\nAdapted instructions preview:")
print(adapted_instructions[-200:])  # Show last 200 chars

Learned preferences: {'prefers_beaches': True}

Adapted instructions preview:
on like "Bali" or "Paris," focus your planning on that location rather than suggesting alternatives.


# Learned User Preferences:
When suggesting destinations, prioritize coastal and beach locations.


In [15]:
# 🔧 Implementing Core Principle: Transparent & Controllable Travel Agent

# Agent Activity Logger - shows what the agent is doing
class AgentActivityLogger:
    def __init__(self):
        self.activities = []
    
    def log_tool_use(self, tool_name: str, purpose: str, result_summary: str):
        """Log when and why tools are used"""
        self.activities.append({
            "timestamp": datetime.now().isoformat(),
            "type": "tool_use",
            "tool": tool_name,
            "purpose": purpose,
            "result": result_summary[:100]  # Truncate for display
        })
    
    def log_memory_access(self, memory_type: str, query: str, findings: str):
        """Log memory system access"""
        self.activities.append({
            "timestamp": datetime.now().isoformat(),
            "type": "memory_access",
            "memory_type": memory_type,
            "query": query,
            "findings": findings[:100]
        })
    
    def get_recent_activities(self, count: int = 5) -> list:
        """Get recent activities for transparency"""
        return self.activities[-count:]

# Confidence scoring for travel recommendations
class ConfidenceCalculator:
    @staticmethod
    def calculate_destination_confidence(destination: str, user_context: dict) -> float:
        """Calculate confidence in a destination recommendation"""
        confidence = 0.5  # Base confidence
        
        # Higher confidence if we have rich data about destination
        if destination in travel_graph.connections:
            confidence += 0.3
        
        # Higher confidence if it matches user preferences
        if user_context.get("prefers_beaches") and "beach" in destination.lower():
            confidence += 0.2
        if user_context.get("likes_history") and destination in ["Paris, France", "Cairo, Egypt"]:
            confidence += 0.2
        
        return min(confidence, 1.0)  # Cap at 1.0
    
    @staticmethod
    def confidence_to_language(score: float) -> str:
        """Convert confidence score to human language"""
        if score >= 0.8:
            return "I'm very confident this is a great match"
        elif score >= 0.6:
            return "I think this would be a good choice"
        elif score >= 0.4:
            return "This might work for you"
        else:
            return "I'm less certain about this suggestion"

# User Control Settings
class TravelAgentController:
    def __init__(self):
        self.settings = {
            "verbosity": "normal",  # concise, normal, detailed
            "show_confidence": True,
            "show_reasoning": False,
            "auto_suggest_connections": True,
            "remember_preferences": True,
            "max_suggestions": 3
        }
    
    def update_settings(self, **updates):
        """Allow user to control agent behavior"""
        for key, value in updates.items():
            if key in self.settings:
                self.settings[key] = value
                print(f"✓ Updated {key} to {value}")
            else:
                print(f"✗ Unknown setting: {key}")
    
    def get_user_control_panel(self) -> str:
        """Show available controls to user"""
        controls = []
        for setting, value in self.settings.items():
            controls.append(f"  {setting}: {value}")
        return "🎛️ Agent Controls:\n" + "\n".join(controls)

# Initialize our enhanced systems
activity_logger = AgentActivityLogger()
confidence_calc = ConfidenceCalculator()
agent_controller = TravelAgentController()

# Enhanced destination recommendation with transparency
def get_transparent_destination(user_preferences: dict = None) -> dict:
    """Get destination with full transparency"""
    user_prefs = user_preferences or {}
    
    # Log the tool use
    activity_logger.log_tool_use(
        "get_random_destination", 
        "Finding destination suggestion for user",
        "Selected destination from available options"
    )
    
    destination = get_random_destination()
    confidence = confidence_calc.calculate_destination_confidence(destination, user_prefs)
    confidence_text = confidence_calc.confidence_to_language(confidence)
    
    return {
        "destination": destination,
        "confidence_score": confidence,
        "confidence_explanation": confidence_text,
        "reasoning": f"Selected {destination} based on available options and user preferences",
        "transparency_available": True
    }

# Test the transparent system
print("🎛️ User Control Panel:")
print(agent_controller.get_user_control_panel())

print("\n🔍 Testing transparent destination selection:")
result = get_transparent_destination({"prefers_beaches": True, "likes_history": True})
print(f"Destination: {result['destination']}")
print(f"Confidence: {result['confidence_explanation']} ({result['confidence_score']:.2f})")
print(f"Reasoning: {result['reasoning']}")

print("\n📋 Recent Agent Activities:")
for activity in activity_logger.get_recent_activities():
    print(f"  {activity['type']}: {activity.get('tool', activity.get('memory_type', 'unknown'))}")

🎛️ User Control Panel:
🎛️ Agent Controls:
  verbosity: normal
  show_confidence: True
  show_reasoning: False
  auto_suggest_connections: True
  remember_preferences: True
  max_suggestions: 3

🔍 Testing transparent destination selection:
Destination: Rio de Janeiro, Brazil
Confidence: This might work for you (0.50)
Reasoning: Selected Rio de Janeiro, Brazil based on available options and user preferences

📋 Recent Agent Activities:
  tool_use: get_random_destination


In [17]:
# 🔧 Creating Enhanced Travel Agent with All Design Principles

# Enhanced destination tool that incorporates all our principles
def get_enhanced_travel_suggestion(user_input: str) -> dict:
    """Enhanced travel suggestion incorporating all design principles"""
    
    # SPACE: Get destination with connection awareness
    base_dest = get_random_destination()
    connections = travel_graph.get_connections(base_dest)
    
    # TIME: Use memory to personalize (access global instance)
    context = travel_memory.get_relevant_context(user_input)
    
    # CORE: Calculate confidence and log activity
    user_prefs = travel_memory.user_preferences
    confidence = confidence_calc.calculate_destination_confidence(base_dest, user_prefs)
    
    activity_logger.log_tool_use(
        "enhanced_travel_suggestion",
        f"Personalized suggestion based on: {user_input}",
        f"Suggested {base_dest} with {len(connections)} connections"
    )
    
    # CONSISTENCY: Format response uniformly
    response = {
        "destination": base_dest,
        "confidence": {
            "score": confidence,
            "explanation": confidence_calc.confidence_to_language(confidence)
        },
        "connections": connections,
        "personalization": context,
        "transparency": {
            "reasoning": f"Selected based on your preferences and available rich information",
            "tool_used": "enhanced_travel_suggestion",
            "memory_accessed": bool(context)
        }
    }
    
    return response

# Enhanced instructions that incorporate all principles
ENHANCED_AGENT_INSTRUCTIONS = """You are an enhanced TravelAgent that embodies human-centric design principles.

CORE BEHAVIORS:
- Connect users to relevant people, events, and knowledge about destinations
- Learn from past conversations and adapt recommendations  
- Be transparent about confidence levels and reasoning
- Give users control over the experience
- Provide consistent, helpful responses

When suggesting destinations:
1. Consider the user's past preferences and conversation history
2. Highlight relevant connections (events, local experts, hidden gems) when available
3. Express your confidence level in the recommendation
4. Explain your reasoning if requested
5. Offer related suggestions that connect to their interests

Always prioritize user preferences. Be helpful but not overwhelming. Remain accessible but unobtrusive.
"""

# Create enhanced agent with all our improvements
enhanced_agent_chat_client = AzureAIAgentClient(
    async_credential=AzureCliCredential(), 
    model_deployment_name="gpt-4o", 
    project_endpoint="https://ibecfoundry.services.ai.azure.com/api/projects/firstProject"
)

enhanced_agent = ChatAgent(
    name="EnhancedTravelAgent",
    chat_client=enhanced_agent_chat_client,
    instructions=ENHANCED_AGENT_INSTRUCTIONS,
    tools=[get_enhanced_travel_suggestion]  # Using our enhanced tool
)

print("✅ Enhanced Travel Agent created successfully!")
print("\n🆚 Comparison Summary:")
print("Original Agent: Basic destination suggestions")
print("Enhanced Agent: + Connections + Memory + Transparency + User Control")

# Demo the enhanced functionality
print("\n🔍 Enhanced suggestion demo:")
sample_suggestion = get_enhanced_travel_suggestion("I want a cultural trip")
print(f"Destination: {sample_suggestion['destination']}")
print(f"Confidence: {sample_suggestion['confidence']['explanation']}")
if sample_suggestion['connections']:
    print(f"Available connections: {len(sample_suggestion['connections'])} categories")
print(f"Transparency: Tool used with reasoning provided")

✅ Enhanced Travel Agent created successfully!

🆚 Comparison Summary:
Original Agent: Basic destination suggestions
Enhanced Agent: + Connections + Memory + Transparency + User Control

🔍 Enhanced suggestion demo:
Destination: Barcelona, Spain
Confidence: I'm very confident this is a great match
Available connections: 3 categories
Transparency: Tool used with reasoning provided


## 🎯 Workshop Reflection & Next Steps

**What We Accomplished:**

✅ **Agent (Space)**: Connected destinations to events, people, and knowledge  
✅ **Agent (Time)**: Added memory, learning, and adaptation capabilities  
✅ **Agent (Core)**: Built transparency, confidence scoring, and user control  
✅ **Integration**: Created an enhanced agent that embodies all principles  

**Key Takeaways:**

1. **Human-Centric Design**: Agents should enhance human capabilities, not replace human judgment
2. **Progressive Enhancement**: Start simple, then add principles incrementally
3. **Trust Through Transparency**: Users trust what they can understand and control
4. **Learning Over Time**: Agents become more valuable as they learn user preferences

**Try These Exercises:**

1. **Test Both Agents**: Compare responses from the original vs enhanced agent
2. **Experiment with Memory**: Add more interactions to see how learning improves suggestions
3. **Adjust Controls**: Use the agent controller to modify behavior and observe changes
4. **Extend Connections**: Add more destinations and connections to the travel graph

**Next Level Enhancements:**

- Persist memory across sessions using a database
- Add real-time event data integration
- Implement collaborative filtering for user preferences
- Build a web interface that shows transparency features
- Add support for group travel planning (multi-user connections)

**Remember**: Great agents don't just answer questions—they help users discover new possibilities while respecting their autonomy and preferences.